# SEAL — R1-Distill-1.5B / GSM8K (run + visualize)

Reproduce SEAL's headline result and present it: **accuracy held/up while reasoning tokens drop**.

**Recommended flow on a RunPod A100 pod:**
1. In a **terminal**: `cp .env.example .env` (add `HF_TOKEN`), then `bash setup_runpod.sh`.
2. Run the generation pipeline. The full run takes hours, so run it in the **terminal** (a notebook kernel disconnect would kill it):
   `bash scripts/runpod_gsm.sh`  — or do a quick `GSM_MAX_EXAMPLES=100 bash scripts/runpod_gsm.sh` first.
3. Come back to **this notebook** and run the *Visualize* cells below to render the charts inline.

You *can* also kick off the pipeline from the notebook (cell in Step 1), but the terminal is safer for the multi-hour run.

## 0. Environment

In [ ]:
import os, sys, subprocess, pathlib

# Point this at the repo root (where scripts/ and visualize_results.py live).
REPO_DIR = pathlib.Path.cwd()
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

# Cache models/datasets on the persistent volume so re-runs don't re-download.
os.environ.setdefault("HF_HOME", "/workspace/hf_cache")
# os.environ["HF_TOKEN"] = "hf_xxx"   # optional; the 1.5B model is public

MODEL       = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
MODEL_TAG   = MODEL.split("/")[-1]
MAX_TOKENS  = 10000
BASELINE_DIR = f"results/GSM/{MODEL_TAG}/baseline_{MAX_TOKENS}"
STEERED_DIR  = f"results/GSM/{MODEL_TAG}/steer-from-MATH-{MAX_TOKENS}"
FIGURES_DIR  = f"results/GSM/{MODEL_TAG}/figures"
print("repo:", REPO_DIR)
print("baseline:", BASELINE_DIR)
print("steered :", STEERED_DIR)

In [ ]:
# Optional: install deps from inside the notebook (skip if you ran setup_runpod.sh).
# %pip install -r requirements.txt
# %pip install --no-deps latex2sympy2==1.9.1

## 1. Run the pipeline  *(optional from here — terminal recommended)*

Three phases: build steering vector from MATH-train (one-time) → baseline GSM8K → steered GSM8K → auto-visualize.
Uncomment to run. Start with the **100-example smoke test**.

In [ ]:
# --- Smoke test (≈30-45 min incl. one-time vector build) ---
# !GSM_MAX_EXAMPLES=100 bash scripts/runpod_gsm.sh

# --- Full GSM8K test set (1319 problems, several hours) ---
# !bash scripts/runpod_gsm.sh

# Tip: in a terminal use `tee` to keep a log:
#   bash scripts/runpod_gsm.sh 2>&1 | tee run.log

## 2. Visualize the results (inline)

Loads each run's `math_eval.jsonl`, counts tokens with the model tokenizer, and renders the dashboard inline.
Pass `model=None` to skip the tokenizer and use word counts instead.

In [ ]:
%matplotlib inline
import importlib, visualize_results as vr
importlib.reload(vr)

summary, fig = vr.visualize(
    baseline_dir=BASELINE_DIR,
    steered_dir=STEERED_DIR,
    model=MODEL,            # set to None for word counts (no tokenizer download)
    dataset="GSM8K",
    output_dir=FIGURES_DIR, # also saves PNGs + summary.md/.json
)
fig  # dashboard renders inline

In [ ]:
# Pretty summary table
from IPython.display import Markdown, display
a = vr.analyze(BASELINE_DIR, STEERED_DIR, model=MODEL, dataset="GSM8K")
display(Markdown(vr.summary_markdown(a)))

In [ ]:
# Individual charts inline (useful when you want one figure per slide)
import matplotlib.pyplot as plt
for title, plot in [("accuracy", vr.plot_accuracy),
                    ("tokens", vr.plot_tokens),
                    ("distribution", vr.plot_distribution),
                    ("efficiency", vr.plot_efficiency)]:
    f, ax = plt.subplots(figsize=(6, 4))
    plot(ax, a)
    plt.show()

## 3. What to look for
- **Accuracy** baseline vs steered should be ~equal or slightly higher.
- **Generation length** should drop noticeably (the SEAL win) — the `-NN%` in the title.
- The **efficiency frontier** arrow should point up-and-left (cheaper *and* at least as accurate).

Saved artifacts live in `results/GSM/<model>/figures/` (`dashboard.png`, `summary.md`, ...).
If the reduction looks small, sweep `STEER_COEF` (e.g. `STEER_COEF=-2.0 bash scripts/runpod_gsm.sh`).